## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
## add your code here

## B 长跑

In [ ]:
## add your code here
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <stdbool.h>

#define MAXN 20010
#define INF 0x3f3f3f3f

typedef struct {
    int pos, cost;
} Point;

Point pts[MAXN];
int dp[MAXN];
int q[MAXN], head, tail;

int cmp(const void *a, const void *b) {
    Point *pa = (Point*)a;
    Point *pb = (Point*)b;
    if (pa->pos != pb->pos)
        return pa->pos - pb->pos;
    return pa->cost - pb->cost;
}

int main() {
    int N, L, Maxn, S;
    while (scanf("%d %d %d %d", &N, &L, &Maxn, &S) == 4) {
        int m = 0;
        for (int i = 0; i < N; ++i) {
            int p, c;
            scanf("%d %d", &p, &c);
            if (p >= 0 && p <= L) {
                pts[m].pos = p;
                pts[m].cost = c;
                m++;
            }
        }
        pts[m].pos = 0; pts[m].cost = 0; m++;
        pts[m].pos = L; pts[m].cost = 0; m++;
        qsort(pts, m, sizeof(Point), cmp);

        int cnt = 0;
        for (int i = 0; i < m; ++i) {
            if (i == 0 || pts[i].pos != pts[cnt-1].pos) {
                pts[cnt++] = pts[i];
            } else {
                if (pts[i].cost < pts[cnt-1].cost)
                    pts[cnt-1].cost = pts[i].cost;
            }
        }
        m = cnt;

        memset(dp, 0x3f, sizeof(int) * m);
        dp[0] = 0;
        head = tail = 0;
        q[tail++] = 0;

        bool ok = false;
        for (int i = 1; i < m; ++i) {
            while (head < tail && pts[i].pos - pts[q[head]].pos > Maxn)
                head++;
            if (head == tail) break;

            int add = (pts[i].pos == L) ? 0 : pts[i].cost;
            dp[i] = dp[q[head]] + add;

            if (pts[i].pos == L && dp[i] <= S) {
                ok = true;
                break;
            }

            while (head < tail && dp[q[tail-1]] >= dp[i])
                tail--;
            q[tail++] = i;
        }

        if (L <= Maxn && 0 <= S) ok = true;

        puts(ok ? "Yes" : "No");
    }
    return 0;
}

## C 最长回文

In [ ]:
## add your code here
#include <stdio.h>
#include <string.h>
#include <stdint.h>

typedef unsigned long long ULL;

#define MAXN 100000
#define MAXM (2 * MAXN + 5)

static char A[MAXN + 5], B[MAXN + 5];
static char TA[MAXM + 5], TB[MAXM + 5];
static int PA[MAXM + 5], PB[MAXM + 5];
static ULL POWB[MAXM + 5], PRE[MAXM + 5], SUF[MAXM + 5];

static int max_int(int a, int b) { return a > b ? a : b; }
static int min_int(int a, int b) { return a < b ? a : b; }

static int build_transformed(const char *s, int n, char *t) {
    int idx = 0;
    t[idx++] = '$';
    t[idx++] = '#';
    for (int i = 0; i < n; ++i) {
        t[idx++] = s[i];
        t[idx++] = '#';
    }
    t[idx] = '@';
    return idx;
}

static int manacher(const char *t, int m, int *p) {
    int mx = 0, id = 0, best = 0;
    for (int i = 1; i < m; ++i) {
        if (mx > i) {
            int mirror = 2 * id - i;
            p[i] = p[mirror] < (mx - i) ? p[mirror] : (mx - i);
        } else {
            p[i] = 1;
        }
        while (t[i + p[i]] == t[i - p[i]]) ++p[i];
        if (i + p[i] > mx) {
            mx = i + p[i];
            id = i;
        }
        if (p[i] - 1 > best) best = p[i] - 1;
    }
    return best;
}

static void init_hash(const char *ta, const char *tb, int m) {
    const ULL BASE = 1315423911ULL;

    POWB[0] = 1;
    for (int i = 1; i <= m; ++i) POWB[i] = POWB[i - 1] * BASE;

    PRE[0] = (unsigned char)ta[0];
    for (int i = 1; i < m; ++i) {
        PRE[i] = PRE[i - 1] * BASE + (unsigned char)ta[i];
    }

    SUF[m - 1] = (unsigned char)tb[m - 1];
    for (int i = m - 2; i >= 0; --i) {
        SUF[i] = SUF[i + 1] * BASE + (unsigned char)tb[i];
    }
}

static ULL get_hash_pre(int l, int r) {
    if (l > r) return 0;
    if (l == 0) return PRE[r];
    return PRE[r] - PRE[l - 1] * POWB[r - l + 1];
}

static ULL get_hash_rev(int l, int r, int m) {
    if (l > r) return 0;
    if (r == m - 1) return SUF[l];
    return SUF[l] - SUF[r + 1] * POWB[r - l + 1];
}

int main(void) {
    int n;
    if (scanf("%d", &n) != 1) return 0;
    scanf("%s", A);
    scanf("%s", B);

    int m = build_transformed(A, n, TA);
    build_transformed(B, n, TB);

    int ans = 0;
    ans = max_int(ans, manacher(TA, m, PA));
    ans = max_int(ans, manacher(TB, m, PB));

    init_hash(TA, TB, m);

    for (int i = 2; i < m; ++i) {
        int base = PA[i];
        if (i - 2 >= 0 && PB[i - 2] > base) base = PB[i - 2];

        int sa = i - base + 1;
        int sb = i - 2 + base - 1;

        int l = 0, r = min_int(sa, m - 1 - sb), add = 0;
        while (l <= r) {
            int mid = (l + r) >> 1;

            int l1 = sa - mid, r1 = sa - 1;
            int l2 = sb + 1,  r2 = sb + mid;

            if (get_hash_pre(l1, r1) == get_hash_rev(l2, r2, m)) {
                add = mid;
                l = mid + 1;
            } else {
                r = mid - 1;
            }
        }

        int cur = (base - 1) + add;
        if (cur > ans) ans = cur;
    }

    printf("%d\n", ans);
    return 0;
}

## D 优惠券

In [ ]:
## add your code here
#include <iostream>
#include <set>
#include <cstring>
#include <string>
#include <fstream>
using namespace std;

const int MAXN = 100010;
int lastOp[MAXN];
int lastPos[MAXN];

int main(int argc, char* argv[]) {
    ios::sync_with_stdio(false);
    cin.tie(0);
    
    ifstream fin;
    if (argc > 1) {
        fin.open(argv[1]);
    }
    
    istream& in = (argc > 1) ? fin : cin;
    
    int m;
    while (in >> m) {
        if (m == 0) {
            cout << "-1\n";
            continue;
        }
        
        memset(lastOp, 0, sizeof(lastOp));
        memset(lastPos, 0, sizeof(lastPos));
        set<int> qPos;
        int ans = -1;
        
        for (int i = 1; i <= m; ++i) {
            string op;
            in >> op;
            
            if (op == "?" || op == "？") {
                qPos.insert(i);
            } else {
                int x;
                in >> x;
                
                if (ans != -1) continue;
                
                if (op == "I") {
                    if (lastOp[x] == 1) {
                        auto it = qPos.upper_bound(lastPos[x]);
                        if (it != qPos.end() && *it < i) {
                            qPos.erase(it);
                        } else {
                            ans = i;
                        }
                    }
                    lastOp[x] = 1;
                    lastPos[x] = i;
                } else {
                    if (lastOp[x] == 0) {
                        if (qPos.empty()) {
                            ans = i;
                        } else {
                            qPos.erase(qPos.begin());
                            lastOp[x] = 2;
                            lastPos[x] = i;
                        }
                    } else if (lastOp[x] == 2) {
                        auto it = qPos.upper_bound(lastPos[x]);
                        if (it != qPos.end() && *it < i) {
                            qPos.erase(it);
                            lastOp[x] = 2;
                            lastPos[x] = i;
                        } else {
                            ans = i;
                        }
                    } else {
                        lastOp[x] = 2;
                        lastPos[x] = i;
                    }
                }
            }
        }
        
        cout << ans << "\n";
    }
    
    return 0;
}

## E 任意点

In [ ]:
## add your code here
#include <stdio.h>
#include <string.h>

#define MAX 1005

int parent[2 * MAX];
int x[105], y[105];

int find(int x) {
    if (parent[x] != x) parent[x] = find(parent[x]);
    return parent[x];
}

void union_set(int a, int b) {
    int ra = find(a), rb = find(b);
    if (ra != rb) parent[ra] = rb;
}

int main() {
    int n;
    scanf("%d", &n);
    
    for (int i = 0; i < 2 * MAX; i++) parent[i] = i;
    
    for (int i = 0; i < n; i++) {
        scanf("%d %d", &x[i], &y[i]);
        union_set(x[i], y[i] + 1000);
    }
    
    int mark[2 * MAX] = {0};
    for (int i = 0; i < n; i++) {
        mark[find(x[i])] = 1;
        mark[find(y[i] + 1000)] = 1;
    }
    
    int cnt = 0;
    for (int i = 0; i < 2 * MAX; i++) {
        if (mark[i]) cnt++;
    }
    
    if (cnt == 0) printf("0\n");
    else printf("%d\n", cnt - 1);
    
    return 0;
}

## F 通配符匹配

In [ ]:
## add your code here
#include <stdio.h>
#include <stdbool.h>
#include <string.h>

#define MAXLEN 100005
#define MAXSEG 15

char pat[MAXLEN];
char str[MAXLEN];
int n;

typedef struct {
    char s[MAXLEN];
    int len;
    int first_non_q;
    int last_non_q;
} Seg;

Seg segs[MAXSEG];
int seg_cnt;
bool start_star, end_star;

void parse_pattern() {
    int len = strlen(pat);
    start_star = (pat[0] == '*');
    end_star = (pat[len - 1] == '*');
    int i = 0;
    seg_cnt = 0;
    while (i < len) {
        if (pat[i] == '*') {
            i++;
            continue;
        }
        int j = 0;
        while (i < len && pat[i] != '*')
            segs[seg_cnt].s[j++] = pat[i++];
        segs[seg_cnt].s[j] = '\0';
        segs[seg_cnt].len = j;
        int fnq = -1, lnq = -1;
        for (int k = 0; k < j; k++) {
            if (segs[seg_cnt].s[k] != '?') {
                if (fnq == -1) fnq = k;
                lnq = k;
            }
        }
        segs[seg_cnt].first_non_q = fnq;
        segs[seg_cnt].last_non_q = lnq;
        seg_cnt++;
    }
}

int find_seg(const char *text, int tlen, int start, Seg *seg) {
    int plen = seg->len;
    if (start + plen > tlen) return -1;
    int limit = tlen - plen;

    if (seg->first_non_q == -1) {
        return start + plen;
    }

    char first_ch = seg->s[seg->first_non_q];
    int first_off = seg->first_non_q;
    char last_ch = seg->s[seg->last_non_q];
    int last_off = seg->last_non_q;

    for (int pos = start; pos <= limit; pos++) {
        if (text[pos + first_off] != first_ch) continue;
        if (text[pos + last_off] != last_ch) continue;

        int ok = 1;
        for (int k = 0; k < plen; k++) {
            char c = seg->s[k];
            if (c != '?' && c != text[pos + k]) {
                ok = 0;
                break;
            }
        }
        if (ok) return pos + plen;
    }
    return -1;
}

bool check(const char *s) {
    int slen = strlen(s);
    if (seg_cnt == 0) return true;
    if (seg_cnt == 1 && !start_star && !end_star) {
        if (slen != segs[0].len) return false;
        for (int i = 0; i < slen; i++) {
            char c = segs[0].s[i];
            if (c != '?' && c != s[i]) return false;
        }
        return true;
    }
    int min_len = 0;
    for (int i = 0; i < seg_cnt; i++) min_len += segs[i].len;
    if (slen < min_len) return false;
    int pos = 0;
    if (!start_star) {
        if (pos + segs[0].len > slen) return false;
        for (int i = 0; i < segs[0].len; i++) {
            char c = segs[0].s[i];
            if (c != '?' && c != s[pos + i]) return false;
        }
        pos += segs[0].len;
    }
    int seg_idx = start_star ? 0 : 1;
    int last_idx = seg_cnt - (end_star ? 0 : 1);
    while (seg_idx < last_idx) {
        pos = find_seg(s, slen, pos, &segs[seg_idx]);
        if (pos == -1) return false;
        seg_idx++;
    }
    if (!end_star) {
        int last_len = segs[seg_cnt - 1].len;
        if (pos > slen - last_len) return false;
        for (int i = 0; i < last_len; i++) {
            char c = segs[seg_cnt - 1].s[i];
            if (c != '?' && c != s[slen - last_len + i]) return false;
        }
    }
    return true;
}

char input_buf[20000000];
int main() {
    int len = fread(input_buf, 1, sizeof(input_buf), stdin);
    int idx = 0;
    int pi = 0;
    while (input_buf[idx] != '\n') pat[pi++] = input_buf[idx++];
    pat[pi] = '\0';
    idx++;
    n = 0;
    while (input_buf[idx] >= '0' && input_buf[idx] <= '9') {
        n = n * 10 + (input_buf[idx++] - '0');
    }
    idx++;
    parse_pattern();
    for (int i = 0; i < n; i++) {
        int si = 0;
        while (input_buf[idx] != '\n' && input_buf[idx] != '\0')
            str[si++] = input_buf[idx++];
        str[si] = '\0';
        idx++;
        puts(check(str) ? "YES" : "NO");
    }
    return 0;
}

## G 汉诺塔

In [ ]:
## add your code here
#include <stdio.h>
#include <string.h>

char pri[6][3];
int stack[3][35];
int top[3];

void init(int n) {
    top[0] = n;
    for (int i = 0; i < n; i++) {
        stack[0][i] = n - i;
    }
    top[1] = top[2] = 0;
}

long long simulate(int n, int *target) {
    init(n);
    long long steps = 0;
    int last = 0;

    while (1) {
        int moved = 0;
        for (int i = 0; i < 6; i++) {
            int from = pri[i][0] - 'A';
            int to   = pri[i][1] - 'A';
            if (top[from] == 0) continue;
            int disk = stack[from][top[from] - 1];
            if (disk == last) continue;
            if (top[to] > 0 && stack[to][top[to] - 1] < disk)
                continue;

            top[from]--;
            stack[to][top[to]++] = disk;
            last = disk;
            steps++;
            moved = 1;
            break;
        }
        if (!moved) break;

        if (top[0] == 0) {
            if (top[1] == n) { *target = 1; break; }
            if (top[2] == n) { *target = 2; break; }
        }
    }
    return steps;
}

int main() {
    int n;
    scanf("%d", &n);
    for (int i = 0; i < 6; i++) {
        scanf("%s", pri[i]);
    }

    if (n <= 4) {
        int target;
        printf("%lld\n", simulate(n, &target));
        return 0;
    }

    int t1, t2, t3, t4;
    long long f1 = simulate(1, &t1);
    long long f2 = simulate(2, &t2);
    long long f3 = simulate(3, &t3);
    long long f4 = simulate(4, &t4);

    if (f2 == f1) {
        printf("%lld\n", simulate(n, &t1));
        return 0;
    }
    long long k = (f3 - f2) / (f2 - f1);
    long long b = f2 - k * f1;

    if (f4 != k * f3 + b) {
        printf("%lld\n", simulate(n, &t1));
        return 0;
    }

    long long ans = f1;
    for (int i = 2; i <= n; i++) {
        ans = k * ans + b;
    }
    printf("%lld\n", ans);

    return 0;
}

## H 马步距离

In [ ]:
## add your code here
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>

#define MAX 205
#define INF 0x3f3f3f3f

int dx[8] = {1, 2, 2, 1, -1, -2, -2, -1};
int dy[8] = {2, 1, -1, -2, -2, -1, 1, 2};

int dist[MAX][MAX];
int queue[MAX * MAX][2];

int bfs(int a, int b) {
    if (a == 0 && b == 0) return 0;
    
    memset(dist, 0x3f, sizeof(dist));
    dist[100][100] = 0;
    int head = 0, tail = 0;
    queue[tail][0] = 100;
    queue[tail][1] = 100;
    tail++;
    
    while (head < tail) {
        int x = queue[head][0];
        int y = queue[head][1];
        head++;
        
        for (int i = 0; i < 8; i++) {
            int nx = x + dx[i];
            int ny = y + dy[i];
            if (nx < 0 || nx >= MAX || ny < 0 || ny >= MAX) continue;
            if (dist[nx][ny] > dist[x][y] + 1) {
                dist[nx][ny] = dist[x][y] + 1;
                queue[tail][0] = nx;
                queue[tail][1] = ny;
                tail++;
            }
        }
    }
    
    return dist[100 + a][100 + b];
}

int min_moves(int a, int b) {
    a = abs(a);
    b = abs(b);
    if (a < b) {
        int t = a;
        a = b;
        b = t;
    }
    
    if (a <= 50 && b <= 50) {
        return bfs(a, b);
    }
    
    int steps = 0;
    while (a > 50 || b > 50) {
        if (a > b) {
            a -= 2;
            b -= 1;
        } else {
            a -= 1;
            b -= 2;
        }
        steps++;
        // 确保非负
        if (a < 0) a = -a;
        if (b < 0) b = -b;
        if (a < b) {
            int t = a;
            a = b;
            b = t;
        }
    }
    
    return steps + bfs(a, b);
}

int main() {
    int x1, y1, x2, y2;
    scanf("%d %d %d %d", &x1, &y1, &x2, &y2);
    
    int a = x2 - x1;
    int b = y2 - y1;
    
    printf("%d\n", min_moves(a, b));
    
    return 0;
}

## I 直方图最大矩形

In [ ]:
## add your code here
int largestRectangleArea(int* heights, int heightsLen) {
    if (heightsLen == 0) return 0;
    
    int *stack = (int*)malloc((heightsLen + 1) * sizeof(int));
    int top = -1;
    int maxArea = 0;
    
    for (int i = 0; i <= heightsLen; i++) {
        int h = (i == heightsLen) ? 0 : heights[i];
        
        while (top >= 0 && h < heights[stack[top]]) {
            int height = heights[stack[top]];
            top--;
            int width = (top == -1) ? i : (i - stack[top] - 1);
            int area = height * width;
            if (area > maxArea) maxArea = area;
        }
        
        stack[++top] = i;
    }
    
    free(stack);
    return maxArea;
}

## J 消防局的设立

In [ ]:
## add your code here
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

int main() {
    int n;
    scanf("%d", &n);
    if (n == 1) { printf("1\n"); return 0; }
    
    int *p = (int*)malloc((n+1)*sizeof(int));
    int *d = (int*)malloc((n+1)*sizeof(int));
    int *cov = (int*)calloc(n+1, sizeof(int));
    int *cnt = (int*)calloc(n+1, sizeof(int));
    int *a = (int*)malloc((n+1)*sizeof(int));
    
    p[1] = 0;
    for (int i = 2; i <= n; i++) {
        scanf("%d", &a[i]);
        p[i] = a[i];
        cnt[a[i]]++;
    }
    
    int **ch = (int**)malloc((n+1)*sizeof(int*));
    int *pos = (int*)calloc(n+1, sizeof(int));
    for (int i = 1; i <= n; i++) {
        if (cnt[i]) ch[i] = (int*)malloc(cnt[i]*sizeof(int));
        else ch[i] = NULL;
    }
    for (int i = 2; i <= n; i++) ch[a[i]][pos[a[i]]++] = i;
    
    d[1] = 0; int md = 0;
    for (int i = 2; i <= n; i++) {
        d[i] = d[p[i]] + 1;
        if (d[i] > md) md = d[i];
    }
    
    int *dc = (int*)calloc(md+1, sizeof(int));
    for (int i = 1; i <= n; i++) dc[d[i]]++;
    
    int **bkt = (int**)malloc((md+1)*sizeof(int*));
    int *bpos = (int*)calloc(md+1, sizeof(int));
    for (int i = 0; i <= md; i++) {
        if (dc[i]) bkt[i] = (int*)malloc(dc[i]*sizeof(int));
        else bkt[i] = NULL;
    }
    for (int i = 1; i <= n; i++) bkt[d[i]][bpos[d[i]]++] = i;
    
    int ans = 0;
    for (int dep = md; dep >= 0; dep--) {
        for (int j = 0; j < dc[dep]; j++) {
            int u = bkt[dep][j];
            if (!cov[u]) {
                int v = u;
                if (p[u]) {
                    v = p[u];
                    if (p[v]) v = p[v];
                }
                cov[v] = 1;
                if (p[v]) {
                    cov[p[v]] = 1;
                    if (p[p[v]]) cov[p[p[v]]] = 1;
                    for (int k = 0; k < cnt[p[v]]; k++) {
                        int s = ch[p[v]][k];
                        if (s != v) cov[s] = 1;
                    }
                }
                for (int k = 0; k < cnt[v]; k++) {
                    int c = ch[v][k];
                    cov[c] = 1;
                    for (int m = 0; m < cnt[c]; m++) cov[ch[c][m]] = 1;
                }
                ans++;
            }
        }
    }
    printf("%d\n", ans);
    
    free(p); free(d); free(cov); free(cnt); free(a); free(pos); free(dc); free(bpos);
    for (int i = 1; i <= n; i++) if (ch[i]) free(ch[i]);
    free(ch);
    for (int i = 0; i <= md; i++) if (bkt[i]) free(bkt[i]);
    free(bkt);
    return 0;
}